# OODimprovements 03: CWD + Foreground-Dilated KD on the Mosaic-Augmented Set
Same loss recipe as `final_notebooks/10_run_layerkd_dilated_hires` (Neck CWD layers 12/15/18 + 8px foreground-dilated mask-KL), but:
- Trains on `crack500_yolo_augmented` (1896 original crops + 250 mosaic composites, up to 1920x720) instead of only the 640x360-capped crops.
- `imgsz=640` (not 768): matches the native single-tile width so the majority of the set isn't force-upsampled the way notebook 10's 768px runs the original-only set through — see `possibleOODimprovements.md` §1.2 for why 768 alone was only a partial fix.
- Teacher logits merge the original crop-scale set with the native-scale mosaic logits generated in notebook 02.

**Trains from the stock pretrained `yolo11n-seg.pt` backbone, like every other notebook in this project** — no prior experiment's `best.pt` is used as a starting point, so this stays a clean, comparable ablation.

**Requires**: attach BOTH notebook 01's output (`crack500_yolo_augmented`) and notebook 02's output (`data/teacher_logits_box` with mosaic logits merged in).


In [ ]:
# -- Environment & Directory Initialization --
!mkdir -p configs utils distillation scripts checkpoints data/datasets data/teacher_logits_box runs results
!pip install -q ultralytics albumentations pycocotools thop pyyaml pandas tqdm opencv-python Pillow


In [ ]:
%%writefile configs/config.yaml
# ============================================================
# Crack-Distill — Master Config
# ============================================================
# YOU ONLY NEED TO CHANGE:
#   1. data.datasets[0].path  → your crack500_yolo folder
#   2. student.backbone       → yolo11n-seg (demo) or yolo11s-seg (paper)
#   3. train.epochs           → 10 (demo) or 100 (real)
# Everything else leave as is.
# ============================================================

project:
  name: crack-distill
  seed: 42
  output_dir: runs/

# ----------------------------------------------------------
# TASK
# To switch: change type ONLY — everything else adapts.
# Options: instance_seg | semantic_seg | detection
# ----------------------------------------------------------
task:
  type: instance_seg
  num_classes: 1
  class_names:
    - crack

# ----------------------------------------------------------
# DATASET  ← change path here
# ----------------------------------------------------------
data:
  root: data/datasets/
  train_split: 0.8
  val_split: 0.1
  test_split: 0.1
  image_size: 512
  batch_size: 16
  num_workers: 4

  datasets:
    - name: combined
      path: data/datasets/combined_yolo
      format: yolo

# ----------------------------------------------------------
# TEACHER (SAM 2)  ← no changes needed
# ----------------------------------------------------------
teacher:
  model: sam2
  checkpoint: checkpoints/sam2_hiera_large.pt
  config: configs/sam2.1/sam2.1_hiera_l.yaml  # sam2 package config
  device: cuda
  prompt_type: box
  save_logits: true
  logits_dir: data/teacher_logits/
  batch_size: 4

# ----------------------------------------------------------
# STUDENT
# ----------------------------------------------------------
student:
  backbone: yolo11n-seg   # primary student backbone (YOLOv11)
  pretrained: true
  device: "0,1"
  imgsz: 512

# ----------------------------------------------------------
# KD LOSSES  ← no changes needed
# L = L_task + α·L_mask_kd + β·L_feature + γ·L_boundary
# ----------------------------------------------------------
distillation:
  enabled: true
  temperature: 3.7769      # tuned via composite OOD-aware Optuna search

  progressive:
    enabled: true
    stage1_pct: 0.30
    stage2_pct: 0.70

  losses:
    task:
      weight: 1.0
    mask_kd:
      enabled: true
      weight: 0.9612       # tuned KL divergence mask weight
    feature:
      enabled: true
      weight: 1.8658       # tuned intermediate feature alignment weight
      layers: [2, 5, 8]
    boundary:
      enabled: true
      weight: 0.8055       # reduced to prevent over-memorizing crop boundary artifacts

# ----------------------------------------------------------
# TRAINING  ← change epochs here
# ----------------------------------------------------------
train:
  epochs: 150    # 10 for demo, 150 for real training
  lr: 0.001
  lr_scheduler: cosine
  warmup_epochs: 3
  optimizer: AdamW
  weight_decay: 0.0005
  amp: false     # disabled FP16 AMP to prevent loss NaN / FP16 overflow on thin crack datasets (e.g. DeepCrack)
  grad_clip: 10.0
  save_every: 10
  patience: 20

# ----------------------------------------------------------
# EVALUATION  ← no changes needed
# ----------------------------------------------------------
eval:
  primary_metric: mAP50-seg
  metrics:
    - mAP50-seg
    - mAP50-95-seg
    - dice
    - boundary_iou
    - fps
    - latency_ms
    - params_M
    - gflops

# ----------------------------------------------------------
# PAPER EXPERIMENTS  ← no changes needed
# ----------------------------------------------------------
experiments:
  - name: baseline_finetune
    distillation.enabled: false
  - name: pseudo_labels
    distillation.losses.mask_kd.enabled: false
    distillation.losses.feature.enabled: false
    distillation.losses.boundary.enabled: false
  - name: full_kd_box
    distillation.enabled: true
    teacher.logits_dir: data/teacher_logits_box/
  - name: full_kd_centroid
    distillation.enabled: true
    teacher.logits_dir: data/teacher_logits_centroid/
  - name: low_data_5pct
    data.train_fraction: 0.05
  - name: low_data_10pct
    data.train_fraction: 0.10
  - name: low_data_25pct
    data.train_fraction: 0.25
  - name: low_data_50pct
    data.train_fraction: 0.50


In [ ]:
%%writefile utils/__init__.py
# utils package


In [ ]:
%%writefile utils/config_loader.py
"""Config loader — converts YAML to a dot-access object."""

import yaml
from pathlib import Path


class ConfigNode:
    """Dot-access config object. cfg.data.batch_size just works."""

    def __init__(self, d: dict):
        for k, v in d.items():
            if isinstance(v, dict):
                setattr(self, k, ConfigNode(v))
            elif isinstance(v, list):
                setattr(self, k, [
                    ConfigNode(i) if isinstance(i, dict) else i for i in v
                ])
            else:
                setattr(self, k, v)

    def get(self, key, default=None):
        return getattr(self, key, default)

    def keys(self):
        return self.__dict__.keys()

    def values(self):
        return self.__dict__.values()

    def items(self):
        return self.__dict__.items()

    def __getitem__(self, key):
        if hasattr(self, key):
            return getattr(self, key)
        raise KeyError(key)

    def __setitem__(self, key, value):
        setattr(self, key, value)

    def __contains__(self, key):
        return hasattr(self, key)

    def __repr__(self):
        return f"ConfigNode({self.__dict__})"

    def __iter__(self):
        return iter(self.__dict__.keys())

    def dict(self):
        result = {}
        for k, v in self.__dict__.items():
            if isinstance(v, ConfigNode):
                result[k] = v.dict()
            elif isinstance(v, list):
                result[k] = [i.dict() if isinstance(i, ConfigNode) else i for i in v]
            else:
                result[k] = v
        return result


def load_config(path: str) -> ConfigNode:
    """Load YAML config and return dot-access ConfigNode."""
    with open(path) as f:
        raw = yaml.safe_load(f)
    return ConfigNode(raw)


def override_config(cfg: ConfigNode, overrides: dict) -> ConfigNode:
    """
    Apply flat-key overrides to a config.
    e.g. override_config(cfg, {"distillation.enabled": False})
    """
    raw = cfg.dict()
    for key_path, value in overrides.items():
        parts = key_path.split(".")
        node = raw
        for p in parts[:-1]:
            node = node.setdefault(p, {})
        node[parts[-1]] = value
    return ConfigNode(raw)


In [ ]:
%%writefile distillation/__init__.py
# distillation package


In [ ]:
%%writefile distillation/kd_trainer.py
"""
KD Trainer — correct implementation
=====================================
Processes soft logits and intermediate encoder features from SAM.
Registers hooks and trains 1x1 projection convolutions for feature distillation.
Uses picklable hooks and temporary hook/loss restoration during model saving to prevent pickling errors.
"""

import cv2
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
from pathlib import Path
from collections import OrderedDict
from ultralytics.models.yolo.segment.train import SegmentationTrainer


class KDYOLODataset(torch.utils.data.Dataset):
    """
    Wrapper for YOLO Dataset that preloads SAM teacher logits and features
    inside dataloader worker processes to hide disk I/O latency from GPU training.
    """
    def __init__(self, base_dataset, logits_dir, kd_cfg):
        self.base_dataset = base_dataset
        self.logits_dir = Path(logits_dir)
        self.features_dir = self.logits_dir.parent / "teacher_features"
        self.kd_cfg = kd_cfg

    def __len__(self):
        return len(self.base_dataset)

    def __getitem__(self, idx):
        item = self.base_dataset[idx]
        img_path = item.get("im_file", "")
        if not img_path:
            item["sam_target"] = None
            item["sam_feat"] = None
            return item
            
        stem = Path(img_path).stem

        sam_target = None
        sam_feat = None

        for prefix in [f"crack500_{stem}", f"deepcrack_{stem}", stem]:
            c = self.logits_dir / f"{prefix}_logits.npy"
            f_path = self.features_dir / f"{prefix}_features.npz"

            if c.exists():
                try:
                    raw = np.load(str(c))  # shape (M, 256, 256)
                    if raw.ndim == 2:
                        raw = np.expand_dims(raw, axis=0)
                    sam_target = torch.from_numpy(raw).float()
                except Exception:
                    pass

                if self.kd_cfg.losses.feature.enabled:
                    if not f_path.exists():
                        f_path = self.logits_dir / f"{prefix}_features.npz"
                    if f_path.exists():
                        try:
                            with np.load(str(f_path)) as data:
                                sam_feat = {
                                    "image_embed": torch.from_numpy(data["image_embed"]).float(),
                                    "feat1": torch.from_numpy(data["feat1"]).float()
                                }
                        except Exception:
                            pass
                break

        item["sam_target"] = sam_target
        item["sam_feat"] = sam_feat
        return item

    @property
    def collate_fn(self):
        return self.base_dataset.collate_fn

    def __getattr__(self, name):
        return getattr(self.base_dataset, name)


class ActiveHook:
    """
    A top-level picklable hook callback class.
    Writes features directly to a class attribute to avoid referencing local closures.
    """
    def __init__(self, key):
        self.key = key

    def __call__(self, module, input, output):
        KDSegmentationTrainer.student_features[self.key] = output


class KDSegmentationTrainer(SegmentationTrainer):
    # Class-level attribute to store hooked features safely
    student_features = {}

    def __init__(self, cfg=None, overrides=None, _callbacks=None, logits_dir=None, kd_cfg=None, **kwargs):
        import os
        from pathlib import Path

        # Auto-convert master ConfigNode / master dict into Ultralytics cfg if master config passed directly
        if hasattr(cfg, "student") or (isinstance(cfg, dict) and "student" in cfg):
            master_cfg = cfg
            if kd_cfg is None:
                if hasattr(master_cfg, "distillation"):
                    kd_cfg = master_cfg.distillation
                elif isinstance(master_cfg, dict) and "distillation" in master_cfg:
                    kd_cfg = master_cfg["distillation"]

            if logits_dir is None:
                if hasattr(master_cfg, "teacher") and hasattr(master_cfg.teacher, "logits_dir"):
                    logits_dir = master_cfg.teacher.logits_dir
                elif isinstance(master_cfg, dict) and "teacher" in master_cfg and "logits_dir" in master_cfg["teacher"]:
                    logits_dir = master_cfg["teacher"]["logits_dir"]

            # Backbone model
            model_name = "yolo11n-seg"
            if hasattr(master_cfg, "student") and hasattr(master_cfg.student, "backbone"):
                model_name = master_cfg.student.backbone
            elif isinstance(master_cfg, dict) and "student" in master_cfg and "backbone" in master_cfg["student"]:
                model_name = master_cfg["student"]["backbone"]
            if not str(model_name).endswith(".pt"):
                model_name = f"{model_name}.pt"

            # Data YAML path
            data_path = "data/datasets/crack500_yolo/dataset.yaml"
            if hasattr(master_cfg, "data"):
                if hasattr(master_cfg.data, "datasets") and len(master_cfg.data.datasets) > 0:
                    d_p = master_cfg.data.datasets[0].path
                    candidate = d_p if str(d_p).endswith(".yaml") else os.path.join(d_p, "dataset.yaml")
                    if os.path.exists(candidate):
                        data_path = candidate
                    elif os.path.exists("data/datasets/crack500_yolo/dataset.yaml"):
                        data_path = "data/datasets/crack500_yolo/dataset.yaml"
                elif hasattr(master_cfg.data, "path"):
                    d_p = master_cfg.data.path
                    data_path = d_p if str(d_p).endswith(".yaml") else os.path.join(d_p, "dataset.yaml")
            elif isinstance(master_cfg, dict) and "data" in master_cfg:
                d_dict = master_cfg["data"]
                if "datasets" in d_dict and len(d_dict["datasets"]) > 0:
                    d_p = d_dict["datasets"][0].get("path", data_path) if isinstance(d_dict["datasets"][0], dict) else getattr(d_dict["datasets"][0], "path", data_path)
                    data_path = d_p if str(d_p).endswith(".yaml") else os.path.join(d_p, "dataset.yaml")

            check_data = (overrides.get("data") if overrides and isinstance(overrides, dict) else None) or data_path
            if check_data and not Path(check_data).exists():
                alt = Path("data/datasets/combined_yolo/dataset.yaml")
                if alt.exists():
                    check_data = str(alt)
                    data_path = check_data
                    if overrides and isinstance(overrides, dict) and "data" in overrides:
                        overrides["data"] = check_data
                elif os.path.exists("data/datasets/crack500_yolo/dataset.yaml"):
                    check_data = "data/datasets/crack500_yolo/dataset.yaml"
                    data_path = check_data
                    if overrides and isinstance(overrides, dict) and "data" in overrides:
                        overrides["data"] = check_data

            # Safely fix hardcoded absolute paths in dataset.yaml.
            # The dataset folder may be a symlink to a read-only /kaggle/input path,
            # so we NEVER write to the symlink target.
            # Instead, we copy dataset.yaml to a writable local path and use that copy.
            if check_data and Path(check_data).exists():
                try:
                    import shutil as _shutil
                    yaml_p = Path(check_data)
                    # Resolve the ACTUAL target to check if it is read-only
                    real_yaml = yaml_p.resolve()
                    abs_dset_dir = str(yaml_p.parent.resolve())
                    original_text = real_yaml.read_text()
                    fixed_lines = [f"path: {abs_dset_dir}" if l.strip().startswith("path:") else l for l in original_text.splitlines()]
                    fixed_text = "\n".join(fixed_lines) + "\n"
                    # Only write if the text actually changed
                    if fixed_text != original_text:
                        # Try writing in place (works if writable)
                        try:
                            yaml_p.write_text(fixed_text)
                            print(f"[KD] Fixed dataset.yaml path in-place: {abs_dset_dir}")
                        except OSError:
                            # Read-only filesystem (Kaggle input symlink): copy to local writable dir
                            local_yaml_dir = Path("data/datasets_yaml")
                            local_yaml_dir.mkdir(parents=True, exist_ok=True)
                            local_yaml = local_yaml_dir / yaml_p.parent.name / "dataset.yaml"
                            local_yaml.parent.mkdir(parents=True, exist_ok=True)
                            local_yaml.write_text(fixed_text)
                            data_path = str(local_yaml)
                            if overrides and isinstance(overrides, dict) and "data" in overrides:
                                overrides["data"] = str(local_yaml)
                            print(f"[KD] Read-only symlink — copied fixed dataset.yaml to: {local_yaml}")
                except Exception as e:
                    print(f"[KD Warning] dataset.yaml path fix skipped: {e}")

            proj_name = getattr(getattr(master_cfg, "project", None), "name", "runs")
            exp_name = getattr(getattr(master_cfg, "project", None), "experiment", "exp")

            auto_overrides = {
                "model": model_name,
                "data": data_path,
                "epochs": getattr(getattr(master_cfg, "train", None), "epochs", 150),
                "imgsz": getattr(getattr(master_cfg, "student", None), "imgsz", getattr(getattr(master_cfg, "data", None), "image_size", 512)),
                "batch": getattr(getattr(master_cfg, "data", None), "batch_size", 16),
                "amp": getattr(getattr(master_cfg, "train", None), "amp", False),
                "lr0": getattr(getattr(master_cfg, "train", None), "lr", 0.001),
                "weight_decay": getattr(getattr(master_cfg, "train", None), "weight_decay", 0.0005),
                "project": str(proj_name),
                "name": str(exp_name),
                "exist_ok": True,
                "task": "segment",
            }
            if overrides and isinstance(overrides, dict):
                auto_overrides.update(overrides)

            from ultralytics.cfg import get_cfg
            cfg = get_cfg(overrides=auto_overrides)
            overrides = None

        super().__init__(cfg=cfg, overrides=overrides, _callbacks=_callbacks, **kwargs)

        if logits_dir is None:
            logits_dir = os.environ.get("KD_LOGITS_DIR", "data/teacher_logits/")
                
        if kd_cfg is None:
            kd_config_json = os.environ.get("KD_CONFIG")
            if kd_config_json:
                try:
                    import json
                    from utils.config_loader import ConfigNode
                    kd_cfg = ConfigNode(json.loads(kd_config_json))
                except Exception:
                    pass
            
            if kd_cfg is None:
                # Under DDP, we can load configuration dynamically as a fallback
                from utils.config_loader import load_config
                try:
                    full_cfg = load_config("configs/config.yaml")
                    kd_cfg = full_cfg.distillation
                except Exception:
                    pass

        self.logits_dir  = Path(str(logits_dir))
        self.kd_cfg      = kd_cfg
        
        if kd_cfg is not None:
            self.temperature = float(kd_cfg.temperature)
            self.kd_weight   = float(kd_cfg.losses.boundary.weight)
        else:
            self.temperature = 1.6502
            self.kd_weight   = 2.0569
        
        self._current_paths = []
        self._kd_logged  = False
        self._no_logits_warned = False
        self.kd_losses   = []
        self._sam_targets = {}   # image_stem → soft target tensor (M, 256, 256)
        self._sam_features = {}  # image_stem → dict of features
        self._hook_handles = []

        logit_files = list(self.logits_dir.glob("*.npy"))
        if len(logit_files) == 0:
            for candidate in [Path("/tmp") / self.logits_dir.name, Path("/tmp/teacher_logits_box"), Path("/tmp/teacher_logits")]:
                if candidate.exists():
                    c_files = list(candidate.glob("*.npy"))
                    if len(c_files) > 0:
                        print(f"[KD] Found {len(c_files)} logit files in fallback '{candidate}'. Redirecting logits_dir.")
                        self.logits_dir = candidate
                        logit_files = c_files
                        try:
                            p_local = Path(str(logits_dir))
                            if not p_local.exists() or (p_local.is_dir() and not list(p_local.glob("*.npy"))):
                                if os.path.lexists(p_local):
                                    os.unlink(p_local) if os.path.islink(p_local) else shutil.rmtree(p_local)
                                p_local.parent.mkdir(parents=True, exist_ok=True)
                                os.symlink(candidate, p_local)
                                print(f"[KD] Created symlink: {p_local} -> {candidate}")
                        except Exception as sym_err:
                            print(f"[KD Warning] Could not symlink fallback: {sym_err}")
                        break

        print(f"[KD] logits_dir : {self.logits_dir}")
        print(f"[KD] logit files: {len(logit_files)}")
        print(f"[KD] temperature: {self.temperature}")

        is_kd_enabled = hasattr(self.kd_cfg, "enabled") and getattr(self.kd_cfg, "enabled")
        if is_kd_enabled and len(logit_files) == 0:
            raise RuntimeError(
                f"[KD FATAL ERROR] logits_dir '{self.logits_dir}' contains 0 logit files (*.npy)!\n"
                f"Knowledge distillation cannot proceed without precomputed teacher logits.\n"
                f"Please ensure the teacher logits dataset is attached and linked properly to '{self.logits_dir}'."
            )

    def setup_model(self):
        """Build model, set up projection layers and hooks, and call parent setup."""
        head_idx = 22
        is_freeze_head = hasattr(self.kd_cfg, "progressive") and self.kd_cfg.progressive.get("freeze_head", False)

        if is_freeze_head:
            if self.args.freeze is None:
                self.args.freeze = [head_idx]
            elif isinstance(self.args.freeze, list):
                if head_idx not in self.args.freeze:
                    self.args.freeze.append(head_idx)
            elif isinstance(self.args.freeze, int):
                self.args.freeze = list(range(self.args.freeze))
                if head_idx not in self.args.freeze:
                    self.args.freeze.append(head_idx)
            print(f"[KD] Progressive: Freezing Segment head at index {head_idx}. args.freeze={self.args.freeze}")

        ckpt = super().setup_model()

        if is_freeze_head:
            try:
                from ultralytics.utils.torch_utils import unwrap_model
                model = unwrap_model(self.model)
            except Exception:
                model = self.model

            for idx, module in enumerate(model.model):
                if type(module).__name__ == "Segment":
                    head_idx = idx
                    break

            for name, param in model.named_parameters():
                if f"model.{head_idx}." in name:
                    param.requires_grad = False
            print(f"[KD] Explicitly set requires_grad=False for Segment head parameters (layer {head_idx})")

        self._setup_proj_layers_and_hooks()
        self._patch_model_loss()
        return ckpt

    def _setup_proj_layers_and_hooks(self):
        """
        Dynamically determine student backbone feature shapes, initialize 1x1 convs
        for channel alignment, and register active training hooks.
        """
        try:
            from ultralytics.utils.torch_utils import unwrap_model
            model = unwrap_model(self.model)
        except Exception:
            model = self.model

        # Determine device
        device = next(self.model.parameters()).device

        # Standard layers to monitor (default 12, 15, 18 for PANet Neck if method is cwd, otherwise 2, 5, 8)
        feat_method = getattr(self.kd_cfg.losses.feature, "method", "mse") if hasattr(self.kd_cfg.losses, "feature") else "mse"
        default_layers = [12, 15, 18] if feat_method == "cwd" else [2, 5, 8]
        layers_to_monitor = self.kd_cfg.losses.feature.layers if hasattr(self.kd_cfg.losses.feature, "layers") else default_layers

        captured_shapes = {}
        def temp_hook(layer_idx):
            def hook(module, input, output):
                captured_shapes[layer_idx] = output.shape
            return hook

        hooks = []
        for idx in layers_to_monitor:
            if idx < len(model.model):
                h = model.model[idx].register_forward_hook(temp_hook(idx))
                hooks.append(h)

        # Run dummy forward pass to extract shapes
        dummy_input = torch.zeros((1, 3, self.args.imgsz, self.args.imgsz), device=device)
        model.eval()
        with torch.no_grad():
            try:
                _ = model(dummy_input)
            except Exception as e:
                print(f"[KD] Error during dummy forward pass for shapes: {e}")
        model.train()

        # Remove temporary hooks
        for h in hooks:
            h.remove()

        # Build projection layers
        proj_dict = nn.ModuleDict()
        for idx in layers_to_monitor:
            if idx in captured_shapes:
                in_channels = captured_shapes[idx][1]
                feature_h = captured_shapes[idx][2]
                stride = self.args.imgsz // feature_h
                out_channels = 64 if stride <= 8 else 256
                
                # Cross-Architecture Projector (CAP): 2-stage conv with BatchNorm for CWD, or 1x1 for MSE
                if feat_method == "cwd":
                    proj_dict[f"layer_{idx}"] = nn.Sequential(
                        nn.Conv2d(in_channels, out_channels, kernel_size=1, bias=False),
                        nn.BatchNorm2d(out_channels),
                        nn.GELU(),
                        nn.Conv2d(out_channels, out_channels, kernel_size=3, padding=1, groups=out_channels, bias=False),
                        nn.BatchNorm2d(out_channels)
                    )
                else:
                    proj_dict[f"layer_{idx}"] = nn.Conv2d(in_channels, out_channels, kernel_size=1)
                print(f"[KD] Feature projection layer {idx} ({feat_method}): stride {stride}, channels {in_channels} -> {out_channels}")

        # Register projection layers on the model so they are part of optimizer parameters
        model.add_module("proj_layers", proj_dict)
        self.proj_layers = proj_dict.to(device)

        # Register active training hooks
        self._register_active_hooks(model)

    def _register_active_hooks(self, model):
        """Helper to register forward hooks on target student model layers."""
        self._hook_handles.clear()
        KDSegmentationTrainer.student_features.clear()
        
        feat_method = getattr(self.kd_cfg.losses.feature, "method", "mse") if hasattr(self.kd_cfg.losses, "feature") else "mse"
        default_layers = [12, 15, 18] if feat_method == "cwd" else [2, 5, 8]
        layers_to_monitor = self.kd_cfg.losses.feature.layers if hasattr(self.kd_cfg.losses.feature, "layers") else default_layers
        for idx in layers_to_monitor:
            if idx < len(model.model):
                h = model.model[idx].register_forward_hook(ActiveHook(f"layer_{idx}"))
                self._hook_handles.append(h)
                print(f"[KD] Forward hook registered for layer {idx}")

    def save_model(self):
        """Override save_model to temporarily detach hooks and restore original loss function on both model and EMA model."""
        try:
            from ultralytics.utils.torch_utils import unwrap_model
            model = unwrap_model(self.model)
        except Exception:
            model = self.model

        # Get EMA model if defined
        ema_model = None
        if hasattr(self, "ema") and self.ema is not None and hasattr(self.ema, "ema"):
            try:
                ema_model = unwrap_model(self.ema.ema)
            except Exception:
                ema_model = self.ema.ema

        # Remove hooks on self.model
        for handle in self._hook_handles:
            handle.remove()
        self._hook_handles.clear()

        # Recursively clear forward hooks in all submodules for both models
        for m in [model, ema_model]:
            if m is not None:
                for submodule in m.modules():
                    submodule._forward_hooks.clear()

        # Restore original loss function if patched on both models
        original_loss_restored = False
        for m in [model, ema_model]:
            if m is not None and hasattr(m, "original_loss"):
                m.loss = m.original_loss
                original_loss_restored = True

        # Call original saving logic
        result = super().save_model()

        # Re-patch loss function on training model
        if original_loss_restored:
            self._patch_model_loss()

        # Re-register active hooks on training model
        self._register_active_hooks(model)
        
        # Reset the active hooks registered flag so that they get registered on the active training model again
        self._active_hooks_registered = False
        return result

    def build_dataset(self, img_path: str, mode: str = "train", batch: int | None = None):
        """Build custom KD dataset that wraps the default YOLO dataset."""
        base_dataset = super().build_dataset(img_path, mode, batch)
        if mode != "train":
            return base_dataset
        return KDYOLODataset(base_dataset, self.logits_dir, self.kd_cfg)

    def preprocess_batch(self, batch):
        """Preprocess batch and map preloaded SAM targets/features to GPU."""
        # Ensure active hooks are registered on the active running model (handles DDP deepcopy recreation)
        if not hasattr(self, "_active_hooks_registered") or not self._active_hooks_registered:
            try:
                from ultralytics.utils.torch_utils import unwrap_model
                active_model = unwrap_model(self.model)
            except Exception:
                active_model = self.model
            self._register_active_hooks(active_model)
            self._active_hooks_registered = True

        # Clear student features at the start of batch preprocessing
        KDSegmentationTrainer.student_features.clear()

        # Safety check to verify that all projection layer parameters are in the optimizer
        if not hasattr(self, "_checked_optimizer") and hasattr(self, "optimizer") and self.optimizer is not None:
            self._checked_optimizer = True
            proj_params = set(self.proj_layers.parameters())
            opt_params = set()
            for group in self.optimizer.param_groups:
                for p in group['params']:
                    opt_params.add(p)
            missing = proj_params - opt_params
            if missing:
                print(f"[KD] WARNING: {len(missing)} projection layer parameters are NOT in the optimizer! Training them will have no effect.")
            else:
                print("[KD] Success: All projection layer parameters are in the optimizer and will receive gradients.")

        batch = super().preprocess_batch(batch)
        im_files = batch.get("im_file", [])
        if isinstance(im_files, (str, Path)):
            im_files = [im_files]
        self._current_paths = list(im_files)

        # Retrieve the preloaded SAM targets and features from the batch dict
        sam_targets_list = batch.get("sam_target", [])
        sam_feats_list = batch.get("sam_feat", [])

        self._sam_targets = {}
        self._sam_features = {}

        for idx, img_path in enumerate(self._current_paths):
            stem = Path(img_path).stem
            
            if idx < len(sam_targets_list) and sam_targets_list[idx] is not None:
                # Clean NaNs and Infs to prevent nan mask_kd losses
                self._sam_targets[stem] = torch.nan_to_num(sam_targets_list[idx].to(self.device), nan=0.0, posinf=0.0, neginf=0.0)
                
            if idx < len(sam_feats_list) and sam_feats_list[idx] is not None:
                self._sam_features[stem] = {
                    k: torch.nan_to_num(v.to(self.device), nan=0.0, posinf=0.0, neginf=0.0) for k, v in sam_feats_list[idx].items()
                }

        if self._current_paths and not self._sam_targets and not self._no_logits_warned:
            stems = [Path(p).stem for p in self._current_paths[:3]]
            print(f"[KD] Warning: no SAM logits matched batch stems {stems}. "
                  f"Run: python scripts/generate_teacher_logits.py")
            self._no_logits_warned = True

        return batch

    def _patch_model_loss(self):
        """Patch model.loss() to add KD loss using student predictions."""
        trainer_ref = self

        try:
            from ultralytics.utils.torch_utils import unwrap_model
            model = unwrap_model(self.model)
        except Exception:
            model = self.model

        if not hasattr(model, "original_loss"):
            model.original_loss = model.loss

        original_loss_fn = model.original_loss.__func__ if hasattr(model.original_loss, "__func__") else None

        def patched_loss(self_model, batch, preds=None):
            if preds is None:
                preds = self_model.forward(batch["img"])

            if original_loss_fn is not None:
                base_loss, loss_items = original_loss_fn(self_model, batch, preds)
            else:
                base_loss, loss_items = type(self_model).loss(self_model, batch, preds)

            if not self_model.training:
                return base_loss, loss_items

            kd_losses = trainer_ref._kd_loss_from_preds(preds, batch, self_model)
            
            # Combine losses
            total = base_loss
            for k, v in kd_losses.items():
                total = total + v

            return total, loss_items

        import types
        model.loss = types.MethodType(patched_loss, model)
        print("[KD] model.loss() patched with detailed KD losses ✓")

    def _kd_loss_from_preds(self, preds, batch, model) -> dict:
        """
        Compute KL divergence, boundary, and feature alignment losses.
        """
        kd_losses = {}
        if not self._sam_targets:
            return kd_losses

        try:
            criterion = model.criterion
            preds_parsed = criterion.parse_output(preds)
            
            # Retrieve target assignments
            (fg_mask, target_gt_idx, target_bboxes, _, _), _, _ = criterion.get_assigned_targets_and_loss(preds_parsed, batch)
            
            pred_masks = preds_parsed["mask_coefficient"].permute(0, 2, 1).contiguous()
            proto = preds_parsed["proto"]
            
            loss_mask_kd = torch.tensor(0.0, device=self.device)
            loss_mask_kd_count = 0
            
            loss_affinity = torch.tensor(0.0, device=self.device)
            loss_affinity_count = 0

            loss_boundary = torch.tensor(0.0, device=self.device)
            loss_boundary_count = 0

            # 1. Compute L_mask and L_boundary (Per-instance matched)
            for i, img_path in enumerate(self._current_paths):
                stem = Path(img_path).stem
                if stem not in self._sam_targets:
                    continue
                
                sam_logits = self._sam_targets[stem]
                fg_mask_i = fg_mask[i]
                
                if fg_mask_i.any() and sam_logits.shape[0] > 0:
                    mask_idx = target_gt_idx[i][fg_mask_i]
                    mask_idx = torch.clamp(mask_idx, 0, sam_logits.shape[0] - 1)
                    
                    # Compute student instance predicted mask logits: (N_pos, H_proto, W_proto)
                    pred_coefs = pred_masks[i][fg_mask_i]
                    pred_mask_logits = torch.einsum("in,nhw->ihw", pred_coefs, proto[i])
                    
                    # Extract corresponding SAM teacher logits: (N_pos, 256, 256)
                    sam_logits_matched = sam_logits[mask_idx]
                    
                    # Target resolution (default 256x256, or 512x512 if high_res enabled)
                    target_res = 512 if getattr(self.kd_cfg.losses.mask_kd, "high_res", False) else 256
                    
                    # Resize both to target resolution
                    student_mask_logits_resized = F.interpolate(
                        pred_mask_logits.unsqueeze(1),
                        size=(target_res, target_res),
                        mode="bilinear",
                        align_corners=False
                    ).squeeze(1)
                    
                    sam_logits_matched_resized = F.interpolate(
                        sam_logits_matched.unsqueeze(1),
                        size=(target_res, target_res),
                        mode="bilinear",
                        align_corners=False
                    ).squeeze(1)
                    
                    # Align dtypes to prevent precision/autocast mismatches
                    sam_logits_matched_resized = sam_logits_matched_resized.to(dtype=student_mask_logits_resized.dtype)
                    
                    # L_mask (KL Divergence on Bernoulli soft probabilities)
                    # FIX: clamp logits before sigmoid to prevent log(0) -> NaN
                    if self.kd_cfg.losses.mask_kd.enabled:
                        sam_clamped = torch.clamp(sam_logits_matched_resized / self.temperature, -15.0, 15.0)
                        stu_clamped = torch.clamp(student_mask_logits_resized / self.temperature, -15.0, 15.0)
                        q = torch.sigmoid(sam_clamped)
                        p_log = F.logsigmoid(stu_clamped)
                        inv_q = 1.0 - q
                        inv_p_log = F.logsigmoid(-stu_clamped)
                        
                        kl = q * (torch.log(q + 1e-8) - p_log) + inv_q * (torch.log(inv_q + 1e-8) - inv_p_log)
                        
                        # Foreground-Dilated / Region-Focused Mask-KL (if enabled)
                        use_focused = getattr(self.kd_cfg.losses.mask_kd, "focused", False) or getattr(self.kd_cfg.losses.mask_kd, "foreground_dilated", False)
                        if use_focused:
                            fg_core = (q > 0.35).float().unsqueeze(1)
                            fg_dilated = F.max_pool2d(fg_core, kernel_size=9, stride=1, padding=4).squeeze(1)
                            fg_core = fg_core.squeeze(1)
                            weight_map = torch.where(fg_core > 0, 1.0, torch.where(fg_dilated > 0, 0.5, 0.05))
                            kl_weighted = (kl * weight_map).sum(dim=(-1, -2)) / (weight_map.sum(dim=(-1, -2)) + 1e-6)
                            loss_mask_kd = loss_mask_kd + kl_weighted.mean() * (self.temperature ** 2)
                        else:
                            loss_mask_kd = loss_mask_kd + kl.mean() * (self.temperature ** 2)
                        loss_mask_kd_count += 1

                    # L_affinity (Spatial Pixel Affinity / Directional Gradient KD)
                    affinity_cfg = getattr(self.kd_cfg.losses, "affinity", None)
                    if affinity_cfg and getattr(affinity_cfg, "enabled", False):
                        p_stu = torch.sigmoid(stu_clamped)
                        d_stu_x = p_stu[:, :, 1:] - p_stu[:, :, :-1]
                        d_stu_y = p_stu[:, 1:, :] - p_stu[:, :-1, :]
                        d_tea_x = q[:, :, 1:] - q[:, :, :-1]
                        d_tea_y = q[:, 1:, :] - q[:, :-1, :]
                        loss_aff = F.mse_loss(d_stu_x, d_tea_x.detach()) + F.mse_loss(d_stu_y, d_tea_y.detach())
                        loss_affinity = loss_affinity + loss_aff
                        loss_affinity_count += 1

                    # L_boundary (Per-instance matched boundary weighted loss)
                    if self.kd_cfg.losses.boundary.enabled:
                        sam_soft = torch.sigmoid(sam_logits_matched_resized / self.temperature)
                        bw = (1.0 - torch.abs(sam_soft - 0.5) * 2).detach()
                        stu_clamped_raw = torch.clamp(student_mask_logits_resized, -30.0, 30.0)
                        bce = F.binary_cross_entropy_with_logits(
                            stu_clamped_raw, sam_soft.detach(), reduction="none"
                        )
                        loss_boundary = loss_boundary + (bce * bw).mean()
                        loss_boundary_count += 1

            if self.kd_cfg.losses.mask_kd.enabled and loss_mask_kd_count > 0:
                kd_losses["mask_kd"] = (loss_mask_kd / loss_mask_kd_count) * self.kd_cfg.losses.mask_kd.weight

            affinity_cfg = getattr(self.kd_cfg.losses, "affinity", None)
            if affinity_cfg and getattr(affinity_cfg, "enabled", False) and loss_affinity_count > 0:
                kd_losses["affinity"] = (loss_affinity / loss_affinity_count) * float(getattr(affinity_cfg, "weight", 1.0))
                
            if self.kd_cfg.losses.boundary.enabled and loss_boundary_count > 0:
                kd_losses["boundary"] = (loss_boundary / loss_boundary_count) * self.kd_cfg.losses.boundary.weight

            # 2. Compute L_feature (Scale-matched alignment or CWD)
            if self.kd_cfg.losses.feature.enabled:
                loss_feat = torch.tensor(0.0, device=self.device)
                feat_method = getattr(self.kd_cfg.losses.feature, "method", "mse")
                default_layers = [12, 15, 18] if feat_method == "cwd" else [2, 5, 8]
                layers_to_monitor = self.kd_cfg.losses.feature.layers if hasattr(self.kd_cfg.losses.feature, "layers") else default_layers
                feat_count = 0
                
                # Layer importance weights (P3 = 0.5, P4 = 0.3, P5 = 0.2)
                stage_weights = {12: 0.5, 15: 0.3, 18: 0.2}
                
                for idx in layers_to_monitor:
                    feat_key = f"layer_{idx}"
                    if feat_key in self.student_features and feat_key in self.proj_layers:
                        sf = self.student_features[feat_key]
                        proj = self.proj_layers[feat_key]
                        sf_proj = proj(sf)
                        
                        feature_h = sf.shape[2]
                        stride = self.args.imgsz // feature_h

                        # Map layer stride directly to SAM feature keys & channels (architecture independent)
                        if stride <= 8:
                            target_key = "feat1"
                            out_channels = 64
                        else:
                            target_key = "image_embed"
                            out_channels = 256
                        
                        # Stack SAM features for the batch (per-item spatial alignment before concat)
                        tf_list = []
                        target_h, target_w = sf_proj.shape[2], sf_proj.shape[3]
                        for img_path in self._current_paths:
                            stem = Path(img_path).stem
                            if stem in self._sam_features and target_key in self._sam_features[stem]:
                                tf_item = self._sam_features[stem][target_key]
                                if tf_item.ndim == 3:
                                    tf_item = tf_item.unsqueeze(0)
                                if tf_item.shape[2:] != (target_h, target_w):
                                    tf_item = F.interpolate(tf_item, size=(target_h, target_w), mode="bilinear", align_corners=False)
                                tf_list.append(tf_item)
                            else:
                                tf_list.append(torch.zeros((1, out_channels, target_h, target_w), device=self.device))
                                
                        tf_batch = torch.cat(tf_list, dim=0).to(dtype=sf_proj.dtype)
                        
                        if feat_method == "cwd":
                            # Channel-Wise Distillation (Spatial Softmax per channel + KL Divergence)
                            t_feat = float(getattr(self.kd_cfg.losses.feature, "temperature", 4.0))
                            b, c, h, w = sf_proj.shape
                            s_soft = F.softmax(sf_proj.view(b, c, -1) / t_feat, dim=-1)
                            t_soft = F.softmax(tf_batch.detach().view(b, c, -1) / t_feat, dim=-1)
                            cwd_kl = F.kl_div(s_soft.log(), t_soft, reduction="batchmean") * (t_feat ** 2)
                            w_stage = stage_weights.get(idx, 1.0 / len(layers_to_monitor))
                            loss_feat = loss_feat + w_stage * cwd_kl
                            feat_count += 1
                        else:
                            # Standard normalized per-layer MSE
                            loss_feat = loss_feat + F.mse_loss(sf_proj, tf_batch.detach())
                            feat_count += 1
                    else:
                        if feat_key not in self.student_features and not self._no_logits_warned:
                            print(f"[KD] Warning: Hook feature {feat_key} not found in student_features. "
                                  f"Forward hooks might not be triggering. Skipping feature KD.")
                            self._no_logits_warned = True
                
                if feat_count > 0 and feat_method != "cwd":
                    loss_feat = loss_feat / feat_count
                kd_losses["feature"] = loss_feat * self.kd_cfg.losses.feature.weight

            # Logging demonstration on first pass
            if not self._kd_logged and kd_losses:
                log_strs = [f"{k}: {float(v):.6f}" for k, v in kd_losses.items()]
                print(f"[KD] ✓ KD losses computed: {', '.join(log_strs)}")
                self._kd_logged = True

        except Exception as e:
            if not self._kd_logged:
                print(f"[KD] Warning: Error computing KD loss: {e} — skipping KD this batch")
                import traceback
                traceback.print_exc()
                self._kd_logged = True

        return kd_losses


In [ ]:
%%writefile scripts/convert_crack500_uncropped.py
#!/usr/bin/env python3
"""
Crack500 Uncropped Test/Val → YOLO seg format converter
======================================================
Converts the original uncropped validation and test sets of Crack500.
Handles EXIF orientation for images by rotating the corresponding masks.

Source directories:
  data/datasets/crack500/valdata/   ← contains {stem}.jpg and {stem}_mask.png
  data/datasets/crack500/testdata/  ← contains {stem}.jpg and {stem}_mask.png

Output (YOLO seg format):
  data/datasets/crack500_uncropped_yolo/
  ├── images/
  │   ├── val/
  │   └── test/
  ├── labels/
  │   ├── val/
  │   └── test/
  └── dataset.yaml
"""

import os
import cv2
import numpy as np
import argparse
import shutil
from pathlib import Path
from tqdm import tqdm
from PIL import Image


CLASS_ID = 0        # single class: crack
MIN_AREA = 50       # minimum pixel area to keep an instance
MIN_POINTS = 6      # minimum polygon points (3 coordinate pairs)


def get_exif_rotation(img_path: Path):
    """Retrieve EXIF orientation tag from image."""
    try:
        with Image.open(img_path) as im:
            exif = im.getexif()
            if exif:
                return exif.get(274)  # 274 is the Orientation tag
    except Exception:
        pass
    return None


def rotate_mask_to_match_image(mask: np.ndarray, exif_orientation: int) -> np.ndarray:
    """Rotate mask array to match image rotation applied by cv2.imread based on EXIF."""
    if exif_orientation == 6:
        return cv2.rotate(mask, cv2.ROTATE_90_CLOCKWISE)
    elif exif_orientation == 8:
        return cv2.rotate(mask, cv2.ROTATE_90_COUNTERCLOCKWISE)
    elif exif_orientation == 3:
        return cv2.rotate(mask, cv2.ROTATE_180)
    return mask


def binary_mask_to_yolo_instances(mask_path: str, img_w: int, img_h: int, exif_orientation: int = None) -> list[str]:
    """
    Read binary PNG mask → rotate based on EXIF → split into instances via connectedComponents
    → convert each to normalized YOLO seg polygon string.
    """
    mask = cv2.imread(mask_path, cv2.IMREAD_GRAYSCALE)
    if mask is None:
        return []

    if exif_orientation:
        mask = rotate_mask_to_match_image(mask, exif_orientation)

    # Threshold (Crack500 masks are binary 0/255)
    binary = (mask > 127).astype(np.uint8)

    # Separate touching cracks into individual instances
    num_labels, labels_map = cv2.connectedComponents(binary)

    label_lines = []
    for label_id in range(1, num_labels):      # 0 = background
        instance = (labels_map == label_id).astype(np.uint8)

        if instance.sum() < MIN_AREA:
            continue

        # Find contours for this instance
        contours, _ = cv2.findContours(
            instance, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE
        )

        for contour in contours:
            if len(contour) < MIN_POINTS // 2:
                continue

            # Flatten and normalize to [0, 1]
            pts = contour.squeeze()
            if pts.ndim == 1:
                pts = pts.reshape(1, 2)

            # Simplify contour slightly to reduce file size
            epsilon = 0.002 * cv2.arcLength(contour, True)
            simplified = cv2.approxPolyDP(contour, epsilon, True).squeeze()
            if simplified.ndim == 1:
                simplified = simplified.reshape(1, 2)
            if len(simplified) < 3:
                simplified = pts

            norm = []
            for x, y in simplified:
                norm.append(x / img_w)
                norm.append(y / img_h)

            if len(norm) < MIN_POINTS:
                continue

            coords_str = " ".join(f"{v:.6f}" for v in norm)
            label_lines.append(f"{CLASS_ID} {coords_str}")

    return label_lines


def process_split(src_dir: Path, dst_dir: Path, split_name: str):
    """Process uncropped val or test split."""
    split_dir = src_dir / f"{split_name}data"
    if not split_dir.exists():
        print(f"  [Warning] Directory {split_dir} does not exist, skipping split {split_name}.")
        return 0

    dst_img_dir = dst_dir / "images" / split_name
    dst_lbl_dir = dst_dir / "labels" / split_name

    dst_img_dir.mkdir(parents=True, exist_ok=True)
    dst_lbl_dir.mkdir(parents=True, exist_ok=True)

    # Find all image files (jpg/jpeg/png that do not contain '_mask')
    all_files = sorted(split_dir.iterdir())
    image_files = [
        f for f in all_files 
        if f.suffix.lower() in ('.jpg', '.jpeg', '.png')
        and '_mask' not in f.name.lower()
        and ':Zone.Identifier' not in f.name
    ]

    converted = 0
    skipped = 0

    for img_path in tqdm(image_files, desc=f"  {split_name}", leave=False):
        stem = img_path.stem

        # Find mask (stem + "_mask.png")
        mask_path = split_dir / f"{stem}_mask.png"
        if not mask_path.exists():
            skipped += 1
            continue

        # Read image to get dimensions (matches how cv2.imread auto-rotates it based on EXIF)
        img = cv2.imread(str(img_path))
        if img is None:
            skipped += 1
            continue
        h, w = img.shape[:2]

        # Get EXIF rotation from image
        exif_orientation = get_exif_rotation(img_path)

        # Convert mask to YOLO seg labels (rotating it to match)
        label_lines = binary_mask_to_yolo_instances(str(mask_path), w, h, exif_orientation)

        # Copy image
        dst_img_path = dst_img_dir / img_path.name
        shutil.copy2(img_path, dst_img_path)

        # Write label file (even if empty — YOLO needs it)
        dst_lbl_path = dst_lbl_dir / f"{stem}.txt"
        with open(dst_lbl_path, "w") as f:
            f.write("\n".join(label_lines))

        converted += 1

    print(f"  {split_name}: {converted} images converted, {skipped} skipped")
    return converted


def main():
    parser = argparse.ArgumentParser(description="Convert Crack500 Uncropped splits to YOLO seg format")
    parser.add_argument(
        "--src",
        type=str,
        default="data/datasets/crack500",
        help="Path to crack500 root dir"
    )
    parser.add_argument(
        "--dst",
        type=str,
        default="data/datasets/crack500_uncropped_yolo",
        help="Output directory"
    )
    args = parser.parse_args()

    src = Path(args.src).expanduser().resolve()
    dst = Path(args.dst).expanduser().resolve()

    print(f"[Convert] Source: {src}")
    print(f"[Convert] Output: {dst}")
    print()

    if not src.exists():
        print(f"ERROR: Source directory not found: {src}")
        return

    if dst.exists():
        print(f"[Warning] Output directory exists, clearing: {dst}")
        shutil.rmtree(dst)

    dst.mkdir(parents=True, exist_ok=True)

    counts = {}
    for split in ["val", "test"]:
        n = process_split(src, dst, split)
        counts[split] = n

    # Write dataset_uncropped.yaml
    yaml_content = f"""# Crack500 Uncropped — YOLO seg format
# Auto-generated by convert_crack500_uncropped.py

path: {dst.resolve()}
train: images/val
val:   images/val
test:  images/test

nc: 1
names:
  0: crack

# Stats
# val:   ~{counts.get('val', 0)} images (uncropped)
# test:  ~{counts.get('test', 0)} images (uncropped)
"""
    with open(dst / "dataset.yaml", "w") as f:
        f.write(yaml_content)
    print(f"\n  dataset.yaml written to {dst / 'dataset.yaml'}")
    print(f"[Done] Converted uncropped splits successfully.")


if __name__ == "__main__":
    main()


In [ ]:
# -- Step 1: Link notebook 01 (augmented dataset) + notebook 02 (native teacher logits) outputs --
import os, shutil
from pathlib import Path

def find_and_link(marker_dir_name, dest_path):
    for root, dirs, files in os.walk("/kaggle/input"):
        if marker_dir_name in dirs:
            src = Path(root) / marker_dir_name
            dest = Path(dest_path)
            dest.parent.mkdir(parents=True, exist_ok=True)
            if os.path.lexists(dest):
                os.unlink(dest) if os.path.islink(dest) else shutil.rmtree(dest)
            os.symlink(src, dest)
            print(f"[Link] {src} -> {dest}")
            return True
    return False

ok1 = find_and_link("crack500_yolo_augmented", "data/datasets/crack500_yolo_augmented")
assert ok1, "Attach notebook 01's output (crack500_yolo_augmented) before running."

ok2 = find_and_link("teacher_logits_box", "data/teacher_logits_box")
assert ok2, "Attach notebook 02's output (data/teacher_logits_box with mosaic logits merged) before running."

# Uncropped OOD val set, from the raw dataset attachment (for post-training validation)
input_dir = Path("/kaggle/input/distill_datasetforme")
if not input_dir.exists():
    input_dir = Path("/kaggle/input")
for root, dirs, files in os.walk(str(input_dir)):
    if "valdata" in dirs:
        !python scripts/convert_crack500_uncropped.py --src {root} --dst data/datasets/crack500_uncropped_yolo
        break


In [ ]:
# -- Step 2: Run Training (exp_mosaic_augmented_hires_layerkd_dilated_T3.7769_W0.9612_seed42_150ep) --
import sys
sys.path.insert(0, ".")
from pathlib import Path
from distillation.kd_trainer import KDSegmentationTrainer
from utils.config_loader import load_config, override_config

cfg = load_config("configs/config.yaml")
EXPERIMENT_NAME = "exp_mosaic_augmented_hires_layerkd_dilated_T3.7769_W0.9612_seed42_150ep"

overrides = {
    'distillation.enabled': True,
    'distillation.temperature': 3.7769,
    'distillation.progressive.enabled': False,
    'distillation.losses.task.weight': 1.0,
    'distillation.losses.mask_kd.enabled': True,
    'distillation.losses.mask_kd.weight': 0.9612,
    'distillation.losses.mask_kd.focused': True,
    'distillation.losses.mask_kd.high_res': False,
    'distillation.losses.affinity.enabled': False,
    'distillation.losses.feature.enabled': True,
    'distillation.losses.feature.method': 'cwd',
    'distillation.losses.feature.weight': 0.25,
    'distillation.losses.feature.temperature': 4.0,
    'distillation.losses.feature.layers': [12, 15, 18],
    'distillation.losses.boundary.enabled': False,
    'student.imgsz': 640,
    'data.image_size': 640,
    'data.batch_size': 12,
    'train.lr': 0.001,
    'train.epochs': 150,
    'train.amp': False,
}
overrides["project.name"] = "crack_distill"
overrides["project.experiment"] = EXPERIMENT_NAME
overrides["project.seed"] = 42
overrides["data.datasets"] = [{"name": "crack500_augmented", "path": "data/datasets/crack500_yolo_augmented", "format": "yolo"}]
overrides["teacher.logits_dir"] = "data/teacher_logits_box/"

cfg = override_config(cfg, overrides)

print(f"=== Starting Run: {EXPERIMENT_NAME} ===")
trainer = KDSegmentationTrainer(cfg)
trainer.train()
print("Training completed.")


In [ ]:
# -- Step 3: Validate Best Checkpoint & Export Results --
import glob, json
from pathlib import Path
from ultralytics import YOLO

EXPERIMENT_NAME = "exp_mosaic_augmented_hires_layerkd_dilated_T3.7769_W0.9612_seed42_150ep"
best_pt = glob.glob(f"runs/**/{EXPERIMENT_NAME}*/weights/best.pt", recursive=True)
assert best_pt, f"No checkpoint found for {EXPERIMENT_NAME}!"

model = YOLO(best_pt[0])
print("\n--- In-Domain Cropped Validation ---")
val_metrics = model.val(data="data/datasets/crack500_yolo_augmented/dataset.yaml", split="val", verbose=True)

results = {
    "experiment": EXPERIMENT_NAME,
    "checkpoint": best_pt[0],
    "metrics_indomain": {
        "mask_mAP50": float(val_metrics.seg.map50),
        "mask_mAP50_95": float(val_metrics.seg.map),
        "box_mAP50": float(val_metrics.box.map50),
        "box_mAP50_95": float(val_metrics.box.map),
    },
}

uncropped_yaml = Path("data/datasets/crack500_uncropped_yolo/dataset.yaml")
if uncropped_yaml.exists():
    print("\n--- OOD Uncropped Validation (direct resize) ---")
    ood_metrics = model.val(data=str(uncropped_yaml), split="val", verbose=True)
    results["metrics_ood"] = {
        "ood_mask_mAP50": float(ood_metrics.seg.map50),
        "ood_mask_mAP50_95": float(ood_metrics.seg.map),
    }

print("\n" + "=" * 60)
print(f"RESULT ({EXPERIMENT_NAME}):")
print(f"  In-Domain Mask mAP50: {results['metrics_indomain']['mask_mAP50']:.4f}")
if "metrics_ood" in results:
    print(f"  OOD Direct Mask mAP50: {results['metrics_ood']['ood_mask_mAP50']:.4f}")
print("=" * 60)

out_file = Path(f"/kaggle/working/results/{EXPERIMENT_NAME}.json")
out_file.parent.mkdir(parents=True, exist_ok=True)
with open(out_file, "w") as f:
    json.dump(results, f, indent=2)
print(f"Saved to {out_file}. Attach this notebook's output to 04 for the tiled/Gaussian OOD eval.")
